<a href="https://colab.research.google.com/github/newaiengineer1-dotcom/AI_Resume_Analyzer/blob/main/Hospital_Ai_Asisitant.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [30]:
!python ingest.py

Working directory: /content
Looking for PDFs in: /content/hospital_knowledge_base
Found 12 PDF(s) under 'hospital_knowledge_base'.
  - Loaded ADM-001_Patient_Admission_Policy.pdf (2 page(s) with text) -> department: Admissions
  - Loaded ADM-002_Discharge_Procedure_SOP.pdf (1 page(s) with text) -> department: Admissions
  - Loaded ADM-003_Insurance_Verification_Guidelines.pdf (1 page(s) with text) -> department: Admissions
  - Loaded DEPT-001_Directory_and_Services_Overview.pdf (2 page(s) with text) -> department: Departments
  - Loaded DEPT-002_Interdepartmental_Referral_Protocol.pdf (1 page(s) with text) -> department: Departments
  - Loaded ER-001_Triage_Protocol.pdf (1 page(s) with text) -> department: Emergency
  - Loaded ER-002_Mass_Casualty_Incident_Plan.pdf (1 page(s) with text) -> department: Emergency
  - Loaded ER-003_Trauma_Activation_Criteria.pdf (1 page(s) with text) -> department: Emergency
  - Loaded PS-001_Incident_Reporting_Policy.pdf (1 page(s) with text) -> departme

In [22]:
!zip -r faiss_index.zip faiss_index
from google.colab import files
files.download("faiss_index.zip")

updating: faiss_index/ (stored 0%)
updating: faiss_index/index.faiss (deflated 7%)
updating: faiss_index/metadata.pkl (deflated 68%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [24]:
from pathlib import Path

FAISS_DIR = Path("/content/hospital-ai-assistant/faiss_index")

print("FAISS folder exists:", FAISS_DIR.exists())

if FAISS_DIR.exists():
    print("\nFiles:")
    for f in FAISS_DIR.iterdir():
        print(f.name, "-", f.stat().st_size, "bytes")

FAISS folder exists: False


In [33]:
from pathlib import Path

BASE_DIR = Path("/content/hospital-ai-assistant")
KB_DIR = BASE_DIR / "hospital_knowledge_base"
FAISS_DIR = BASE_DIR / "faiss_index"

KB_DIR.mkdir(parents=True, exist_ok=True)
FAISS_DIR.mkdir(parents=True, exist_ok=True)

print("Knowledge-base files:")

for file in KB_DIR.iterdir():
    print(" -", file.name)

Knowledge-base files:


In [32]:
!pip install -q python-docx

from pathlib import Path
from pypdf import PdfReader
from docx import Document as DocxDocument


def extract_pdf(file_path):

    documents = []

    reader = PdfReader(str(file_path))

    for page_number, page in enumerate(reader.pages, start=1):

        text = page.extract_text() or ""

        if text.strip():

            documents.append({
                "text": text,
                "source": file_path.name,
                "page": page_number,
                "type": "pdf"
            })

    return documents


def extract_docx(file_path):

    doc = DocxDocument(str(file_path))

    text = "\n".join(
        paragraph.text
        for paragraph in doc.paragraphs
        if paragraph.text.strip()
    )

    return [{
        "text": text,
        "source": file_path.name,
        "page": None,
        "type": "docx"
    }]


def extract_text(file_path):

    text = file_path.read_text(
        encoding="utf-8",
        errors="ignore"
    )

    return [{
        "text": text,
        "source": file_path.name,
        "page": None,
        "type": file_path.suffix.lower().replace(".", "")
    }]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 6.3 MB/s eta 0:00:00


In [35]:
from pathlib import Path
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

# Gather and extract all documents from the local folder first
KB_DIR = Path("/content/hospital_knowledge_base")
all_documents = []

if KB_DIR.exists():
    for file_path in KB_DIR.iterdir():
        if file_path.is_file():
            ext = file_path.suffix.lower()
            if ext == ".pdf":
                all_documents.extend(extract_pdf(file_path))
            elif ext == ".docx":
                all_documents.extend(extract_docx(file_path))
            elif ext in [".txt", ".md"]:
                all_documents.extend(extract_text(file_path))

splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=120,
    separators=[
        "\n\n",
        "\n",
        ". ",
        " ",
        ""
    ]
)

chunks = []

for doc in all_documents:

    text_chunks = splitter.split_text(doc["text"])

    for chunk_text in text_chunks:

        if chunk_text.strip():

            chunks.append(
                Document(
                    page_content=chunk_text,
                    metadata={
                        "source": doc["source"],
                        "page": doc["page"],
                        "type": doc["type"]
                    }
                )
            )


for i, chunk in enumerate(chunks):

    chunk.metadata["chunk_id"] = f"chunk_{i:05d}"


print("Total chunks:", len(chunks))

Total chunks: 6


In [37]:
!pip install -q langchain-huggingface

from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={
        "device": "cpu"
    },
    encode_kwargs={
        "normalize_embeddings": True
    }
)

print("Embedding model loaded.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model loaded.


In [39]:
test_vector = embeddings.embed_query(
    "hospital emergency department"
)

print("Embedding dimension:", len(test_vector))

Embedding dimension: 384


In [40]:
from langchain_community.vectorstores import FAISS

print("Chunks available:", len(chunks))

test_chunks = chunks[:3]

print("Creating test FAISS index...")

test_vectorstore = FAISS.from_documents(
    documents=test_chunks,
    embedding=embeddings
)

print("SUCCESS!")
print("FAISS test index created.")

/tmp/ipykernel_515/3043029631.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


Chunks available: 6
Creating test FAISS index...
SUCCESS!
FAISS test index created.


In [41]:
FAISS_DIR.mkdir(parents=True, exist_ok=True)

print("Creating complete FAISS index...")
print("Total chunks:", len(chunks))

vectorstore = FAISS.from_documents(
    documents=chunks,
    embedding=embeddings
)

vectorstore.save_local(
    str(FAISS_DIR)
)

print("FAISS index saved!")

Creating complete FAISS index...
Total chunks: 6
FAISS index saved!


In [42]:
print("FAISS directory:", FAISS_DIR)

for file in FAISS_DIR.iterdir():

    print(
        file.name,
        "->",
        file.stat().st_size,
        "bytes"
    )

FAISS directory: /content/hospital-ai-assistant/faiss_index
index.pkl -> 4294 bytes
index.faiss -> 9261 bytes


In [43]:
index_file = FAISS_DIR / "index.faiss"
pkl_file = FAISS_DIR / "index.pkl"

print("index.faiss exists:", index_file.exists())
print("index.pkl exists:", pkl_file.exists())

if index_file.exists() and pkl_file.exists():
    print("================================")
    print("FAISS READY FOR GITHUB")
    print("================================")

index.faiss exists: True
index.pkl exists: True
FAISS READY FOR GITHUB
